In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [2]:
url = "https://www.ataberkestate.com/turkey/alanya"

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [4]:
print("Завантажуємо сторінку Ataberk Estate...")
response = requests.get(url, headers=headers)

Завантажуємо сторінку Ataberk Estate...


In [5]:
response

<Response [200]>

In [6]:
if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Шукаємо всі картки товарів на сторінці (на цьому сайті вони мають клас 'product-item')
    cards = soup.find_all("div", class_="product-item")
    
    apartments = []
    
    print(f"Знайдено {len(cards)} карток на першій сторінці. Починаємо розбір...")

Знайдено 0 карток на першій сторінці. Починаємо розбір...


In [7]:
print(response.text[:500])


<!DOCTYPE html>
<html lang="ru" class="preload">
<head>
  <meta charset="utf-8" />
<noscript><style>form.antibot * :not(.antibot-message) { display: none !important; }</style>
</noscript><script>(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
j=d.createElement(s),dl=l!='dataLayer'?'&amp;l='+l:'';j.async=true;j.src=
'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
})(window,docu


нас розпізнали як бота, треба щось складніше

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [10]:
# 1. Налаштовуємо прихований браузер Chrome
options = webdriver.ChromeOptions()
# Запускаємо в "headless" режимі (без відкриття вікна на екрані, щоб не заважало)
options.add_argument("--headless") 
options.add_argument("--disable-blink-features=AutomationControlled") # маскуємо, що це робот
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

print("Запускаємо контрольований браузер Chrome...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


Запускаємо контрольований браузер Chrome...


In [11]:
try:
    print("Заходимо на сайт (це може зайняти пару секунд, поки антибот перевіряє нас)...")
    driver.get(url)
    
    # Чекаємо 5 секунд, щоб сторінка повністю завантажилась і відпрацювали всі скрипти
    time.sleep(5)
    
    # Забираємо вже повністю готовий HTML код сторінки з браузера
    html_source = driver.page_source
    
    # Передаємо його в BeautifulSoup, як і раніше
    soup = BeautifulSoup(html_source, "html.parser")
    
    # Спробуємо знайти хоча б посилання на об'єкти, щоб перевірити чи ми всередині
    # На сайтах часто картки лежать в тегах <a> (посиланнях)
    links = soup.find_all("a")
    print(f"Успіх! Браузер обійшов захист. Знайдено елементів на сторінці: {len(links)}")
    
    # Давайте виведемо перші 2000 символів коду тепер, щоб переконатися, що antibot зник
    print("\nШматочок нового коду сайту:")
    print(html_source[:1000])

finally:
    # Обов'язково закриваємо браузер після роботи, щоб він не висів у пам'яті комп'ютера
    driver.quit()
    print("Браузер закрито.")

Заходимо на сайт (це може зайняти пару секунд, поки антибот перевіряє нас)...
Успіх! Браузер обійшов захист. Знайдено елементів на сторінці: 435

Шматочок нового коду сайту:
<html lang="ru" class="js js flexbox canvas canvastext webgl no-touch geolocation postmessage no-websqldatabase indexeddb hashchange history draganddrop websockets rgba hsla multiplebgs backgroundsize borderimage borderradius boxshadow textshadow opacity cssanimations csscolumns cssgradients cssreflections csstransforms csstransforms3d csstransitions fontface generatedcontent video audio localstorage sessionstorage webworkers no-applicationcache no-svg no-inlinesvg no-smil no-svgclippaths" data-once="drupal-dialog-deprecation-listener kibmak2-map-popup-delegated kibmak2-mobile-header-buttons webform-dialog"><head><script src="https://mc.yandex.ru/metrika/tag_phono.js" type="text/javascript" charset="utf-8" async=""></script>
  <meta charset="utf-8">
<noscript><style>form.antibot * :not(.antibot-message) { display: 

In [15]:
# Шукаємо взагалі ВСІ блоки <div> на сторінці
all_divs = soup.find_all("div")
    
# Створюємо список для збереження унікальних назв класів
interesting_classes = set()

for div in all_divs:
    # Перевіряємо, чи є у цього div клас
    if div.has_attr('class'):
        # Перетворюємо список класів у один рядок текст
        class_text = " ".join(div['class'])
        
        # Якщо в назві класу є слова-маркери, які часто дають карткам
        if any(word in class_text.lower() for word in ["product", "item", "card", "estate", "property", "object"]):
            interesting_classes.add(class_text)
            
print("\n--- Знайдені підозрілі класи на сторінці ---")
for cls in sorted(interesting_classes):
    print(cls)


--- Знайдені підозрілі класи на сторінці ---
SumoSelect sumo_field_property_type_target_id
faq-item
faq-item-a
faq-item-q
form-group form-type-object
form-type-object--wrapper
js-form-item form-item form-type-checkbox js-form-type-checkbox form-item-term-node-tid-depth-10 js-form-item-term-node-tid-depth-10
js-form-item form-item form-type-checkbox js-form-type-checkbox form-item-term-node-tid-depth-11 js-form-item-term-node-tid-depth-11
js-form-item form-item form-type-checkbox js-form-type-checkbox form-item-term-node-tid-depth-12 js-form-item-term-node-tid-depth-12
js-form-item form-item form-type-checkbox js-form-type-checkbox form-item-term-node-tid-depth-13 js-form-item-term-node-tid-depth-13
js-form-item form-item form-type-checkbox js-form-type-checkbox form-item-term-node-tid-depth-14 js-form-item-term-node-tid-depth-14
js-form-item form-item form-type-checkbox js-form-type-checkbox form-item-term-node-tid-depth-15 js-form-item-term-node-tid-depth-15
js-form-item form-item fo

In [17]:
# Шукаємо ліві та праві картки об'єктів
left_cards = soup.find_all("div", class_="left-side-card")
right_cards = soup.find_all("div", class_="right-side-card")

# Об'єднуємо їх в один великий список
all_cards = left_cards + right_cards
print(f"\nУра! Знайдено карток нерухомості: {len(all_cards)}")

apartments_list = []

for card in all_cards:
    try:
    # Витягуємо весь чистий текст з картки
        card_text = card.text.strip()
        
        # Щоб текст не йшов одним суцільним рядком, розіб'ємо його по знаку переносу
        # і приберемо порожні рядки
        lines = [line.strip() for line in card_text.split('\n') if line.strip()]
        
        # Додаємо сирі дані в список
        apartments_list.append({
            "raw_text": " | ".join(lines) # з'єднуємо рисочкою для читаємості
        })
    except Exception as e:
            print(f"Помилка при розборі картки: {e}")
            continue
    


Ура! Знайдено карток нерухомості: 16


In [18]:
apartments_list

[{'raw_text': '140 000 € | Комнат:2+1 | Площадь:100 m² | Район: Алания / Центр | До моря:550 м'},
 {'raw_text': '82 500 € | Комнат:1+1 | Площадь:50 m² | Район: Алания / Центр | До моря:550 м'},
 {'raw_text': '283 500 € | Комнат: | Площадь:110 m² | Район: Алания / Центр | До моря:'},
 {'raw_text': '157 500 € | Комнат:1+1 | Площадь:68 m² | Район: Алания / Махмутлар | До моря:600 м'},
 {'raw_text': '86 000 € | Комнат:1+1 | Площадь:55 m² | Район: Алания / Кестель | До моря:627 м'},
 {'raw_text': '74 000 € | Комнат:1+1 | Площадь:55 m² | Район: Алания / Махмутлар | До моря:1,5 км'},
 {'raw_text': '77 500 € | Комнат:1+1 | Площадь:50 m² | Район: Алания / Каргыджак | До моря:200 м'},
 {'raw_text': '350 000 € | Комнат:3+1 | Площадь:200 m² | Район: Алания / Каргыджак | До моря:4 км'},
 {'raw_text': 'Лаконичное решение в центре города - функциональный формат с базовыми, но практичными параметрами. | Видеокамеры | Домофон | Консьерж | Лифт | Настольный теннис | Сад'},
 {'raw_text': 'Центр города, п

In [12]:
# Налаштування прихованого браузера
options = webdriver.ChromeOptions()
options.add_argument("--headless") 
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

all_apartments = []
# Зберемо дані з будь-якої можливої кількості сторінок
total_pages = 270

In [13]:
try:
    for page in range(0, total_pages):
        # Формуємо динамічний URL для кожної сторінки
        url = f"https://www.ataberkestate.com/turkey/alanya?page={page}"
        print(f"\n--- Працюємо зі сторінкою {page + 1} з {total_pages} ---")
        print(f"Заходимо на: {url}")
        
        driver.get(url)
        time.sleep(4) # Чекаємо первинне завантаження
        
        # --- ЕТАП СКРОЛІНГУ (Lazy Loading Bypass) ---
        print("Прокручуємо сторінку вниз для довантаження карток...")
        for scroll in range(4):
            # Магічна команда JavaScript, яка рухає скролл на самий низ сторінки
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            print(f"  Скролл {scroll + 1}/4 виконано...")
            time.sleep(2) # Даємо 2 секунди сайту, щоб він встиг підтягнути нові дані
            
        # Забираємо HTML код сторінки після того, як все довантажилось
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # Шукаємо наші перевірені класи
        left_cards = soup.find_all("div", class_="left-side-card")
        right_cards = soup.find_all("div", class_="right-side-card")
        page_cards = left_cards + right_cards
        
        print(f"На цій сторінці знайдено об'єктів: {len(page_cards)}")
        
        # Збираємо текст з карток поточної сторінки
        for card in page_cards:
            card_text = card.text.strip()
            lines = [line.strip() for line in card_text.split('\n') if line.strip()]
            
            # Додаємо словник у наш загальний список
            all_apartments.append({
                "page": page + 1,
                "raw_text": " | ".join(lines)
            })
            
        # Маленька пауза між сторінками, щоб поводитися чемно щодо сервера сайту
        time.sleep(3)

    # --- ЗБЕРЕЖЕННЯ РЕЗУЛЬТАТІВ ---
    df = pd.DataFrame(all_apartments)
    print(f"\nЗагалом зібрано об'єктів з усіх сторінок: {len(df)}")
    
    # Зберігаємо у файл
    df.to_csv("alanya_big_dataset.csv", index=False, encoding="utf-8")
    print("Великий датасет успішно збережено в 'alanya_big_dataset.csv'!")

except Exception as main_error:
    print(f"Сталася глобальна помилка: {main_error}")

finally:
    driver.quit()
    print("Браузер закрито.")


--- Працюємо зі сторінкою 1 з 270 ---
Заходимо на: https://www.ataberkestate.com/turkey/alanya?page=0
Прокручуємо сторінку вниз для довантаження карток...
  Скролл 1/4 виконано...
  Скролл 2/4 виконано...
  Скролл 3/4 виконано...
  Скролл 4/4 виконано...
На цій сторінці знайдено об'єктів: 16

--- Працюємо зі сторінкою 2 з 270 ---
Заходимо на: https://www.ataberkestate.com/turkey/alanya?page=1
Прокручуємо сторінку вниз для довантаження карток...
  Скролл 1/4 виконано...
  Скролл 2/4 виконано...
  Скролл 3/4 виконано...
  Скролл 4/4 виконано...
На цій сторінці знайдено об'єктів: 16

--- Працюємо зі сторінкою 3 з 270 ---
Заходимо на: https://www.ataberkestate.com/turkey/alanya?page=2
Прокручуємо сторінку вниз для довантаження карток...
  Скролл 1/4 виконано...
  Скролл 2/4 виконано...
  Скролл 3/4 виконано...
  Скролл 4/4 виконано...
На цій сторінці знайдено об'єктів: 16

--- Працюємо зі сторінкою 4 з 270 ---
Заходимо на: https://www.ataberkestate.com/turkey/alanya?page=3
Прокручуємо сто

In [21]:
df.head()

,page,raw_text
0,1,140 000 € | Комнат:2+1 | Площадь:100 m² | Райо...
1,1,82 500 € | Комнат:1+1 | Площадь:50 m² | Район:...
2,1,283 500 € | Комнат: | Площадь:110 m² | Район: ...
3,1,157 500 € | Комнат:1+1 | Площадь:68 m² | Район...
4,1,86 000 € | Комнат:1+1 | Площадь:55 m² | Район:...


In [14]:
df['raw_text'][0]

'140 000 € | Комнат:2+1 | Площадь:100 m² | Район: Алания / Центр | До моря:550 м'